# Polynomial Regression

While standard Linear Regression is powerful, it assumes a straight-line (linear) relationship between features and the target. In the real world, relationships are frequently curved. **Polynomial Regression** solves this limitation by extending linear regression, allowing us to model complex, non-linear relationships without losing the simplicity and speed of linear modeling.

---

## 1. The Need for Polynomial Regression

### 1.1 Limitations of Linear Regression
When the true relationship between a feature $X$ and the target $y$ is curved, a simple linear line:

$$y = \beta_0 + \beta_1 x$$

will underfit the data. It cannot adapt to the curvature, resulting in high bias, low $R^2$ scores, and structured patterns in the residual plot.

### 1.2 The Polynomial Solution
Polynomial Regression transforms the input feature by introducing higher-degree mathematical terms of the inputs (e.g., $x^2, x^3, \dots, x^d$):

$$y = \beta_0 + \beta_1 x + \beta_2 x^2 + \beta_3 x^3 + \dots + \beta_d x^d$$

This formula allows the model to fit curves, waves, and complex non-linear trends depending on the chosen **degree ($d$)** of the polynomial.

### 1.3 The Linearity Paradox: Why is it still "Linear" Regression?
> **Key Insight:** In machine learning, a model is classified as **linear** if it is linear with respect to its **parameters (coefficients $\beta$)**, not the features $X$.

In the polynomial regression equation, the coefficients $\beta_0, \beta_1, \beta_2$ are only added and multiplied by constants (just like in simple linear regression). Mathematically, we can substitute $z_1 = x, z_2 = x^2, z_3 = x^3$, yielding:

$$y = \beta_0 + \beta_1 z_1 + \beta_2 z_2 + \beta_3 z_3$$

This is a standard multiple linear regression equation. Because the model remains linear in terms of its parameters $\beta$, we can train it using OLS (Ordinary Least Squares) or Gradient Descent without modification.

## 2. Practical Implementation: Single Variable

To perform polynomial regression, we run a two-step pipeline:
1. **Feature Transformation:** Map our 1-dimensional feature $X$ to a higher-dimensional space containing powers $[1, x, x^2, \dots, x^d]$ using Scikit-Learn's `PolynomialFeatures` preprocessor.
2. **Linear Regression:** Train a standard `LinearRegression` model on these newly engineered polynomial features.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Set display style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Generate synthetic quadratic dataset (e.g., y = 0.5 * x^2 - x + 2 + noise)
np.random.seed(42)
X = np.random.uniform(-3, 3, 100).reshape(-1, 1)
y = 0.5 * (X.ravel() ** 2) - X.ravel() + 2 + np.random.normal(0, 0.5, 100)

# Visualize raw dataset
plt.figure(figsize=(8, 5))
plt.scatter(X, y, color='#3498db', alpha=0.8, edgecolors='w', label='Raw Data')
plt.title('Synthetic Non-Linear Dataset')
plt.xlabel('Feature X')
plt.ylabel('Target y')
plt.legend()
plt.show()


In [ ]:
# 1. Initialize PolynomialFeatures with degree=2
poly = PolynomialFeatures(degree=2, include_bias=False)

# 2. Transform the input feature X
# If input is [x], the output will be [x, x^2]
X_poly = poly.fit_transform(X)

print("Feature Shapes Comparison:")
print(f"Original X shape:   {X.shape}")
print(f"Transformed X shape: {X_poly.shape}")
print(f"\nFirst 3 rows of original X:\n{X[:3]}")
print(f"First 3 rows of transformed X (x, x^2):\n{X_poly[:3]}")

# 3. Fit a standard OLS Linear Regression model on transformed features
poly_lr = LinearRegression()
poly_lr.fit(X_poly, y)

print(f"\nTrained Intercept (beta_0): {poly_lr.intercept_:.4f}")
print(f"Trained Coefficients (beta_1, beta_2): {poly_lr.coef_}")


In [ ]:
# Generate linear model for baseline comparison
linear_lr = LinearRegression()
linear_lr.fit(X, y)

# Generate test values for smooth curves plotting
X_test = np.linspace(-3, 3, 200).reshape(-1, 1)
X_test_poly = poly.transform(X_test)

y_pred_linear = linear_lr.predict(X_test)
y_pred_poly = poly_lr.predict(X_test_poly)

# Plotting the regression curves
plt.figure(figsize=(10, 6))
plt.scatter(X, y, color='#3498db', alpha=0.7, edgecolors='w', label='Data Points')
plt.plot(X_test, y_pred_linear, color='#e74c3c', linewidth=2.5, linestyle='--', label='Linear Model (Degree 1)')
plt.plot(X_test, y_pred_poly, color='#2ecc71', linewidth=3, label='Polynomial Model (Degree 2)')

plt.title('Linear Regression vs. Polynomial Regression (Degree 2)', fontsize=13, fontweight='bold')
plt.xlabel('Feature X')
plt.ylabel('Target y')
plt.legend()
plt.show()

# Evaluate scores on training set
r2_lin = r2_score(y, linear_lr.predict(X))
r2_poly = r2_score(y, poly_lr.predict(X_poly))

print(f"Linear Model R2 Score:     {r2_lin:.4f}")
print(f"Polynomial Model R2 Score: {r2_poly:.4f} (Significant improvement!)")


## 3. The Bias-Variance Tradeoff (Choosing the Right Degree)

Selecting the optimal degree $d$ is crucial for model generalization. If the degree is incorrect, the model will suffer from either underfitting or overfitting:

### 3.1 Underfitting (High Bias)
* Occurs when the polynomial degree is **too low** (e.g., $d=1$ for curved data).
* The model is too simple to capture the underlying structure, leading to high training error and high test error.

### 3.2 Overfitting (High Variance)
* Occurs when the polynomial degree is **too high** (e.g., $d=15$ or $d=50$ for a quadratic dataset).
* The model behaves like a high-degree curve that weaves through almost every training point, fitting the random noise in the training data rather than the true signal.
* This leads to extremely low training error but massive errors on unseen test data, resulting in poor generalization.

Let's visualize this tradeoff by fitting polynomials of degree $1$, $3$, and $15$ on a noisy dataset.

In [ ]:
# Split data into train and test sets to observe generalization error
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

degrees = [1, 3, 15]  # Underfit, Optimal, Overfit
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharey=True)

X_line = np.linspace(-3, 3, 500).reshape(-1, 1)

for idx, d in enumerate(degrees):
    # Pipeline transformation & training
    poly_features = PolynomialFeatures(degree=d, include_bias=False)
    X_train_poly = poly_features.fit_transform(X_train)
    X_val_poly = poly_features.transform(X_val)
    
    model = LinearRegression()
    model.fit(X_train_poly, y_train)
    
    # Predict for curves and evaluation
    y_line_pred = model.predict(poly_features.transform(X_line))
    r2_tr = r2_score(y_train, model.predict(X_train_poly))
    r2_va = r2_score(y_val, model.predict(X_val_poly))
    
    # Plotting
    axes[idx].scatter(X_train, y_train, color='#3498db', alpha=0.7, label='Train Data')
    axes[idx].scatter(X_val, y_val, color='#f1c40f', marker='s', alpha=0.7, label='Val Data')
    axes[idx].plot(X_line, y_line_pred, color='#e74c3c', linewidth=2.5, label='Fit Model')
    
    title_str = f"Degree {d} "
    if d == 1: title_str += "(Underfit - High Bias)"
    elif d == 3: title_str += "(Optimal Fit)"
    else: title_str += "(Overfit - High Variance)"
    
    axes[idx].set_title(title_str, fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('X')
    if idx == 0:
        axes[idx].set_ylabel('y')
    axes[idx].legend()
    
    # Print performance statistics
    print(f"Degree {d:2d} -> Train R2: {r2_tr:.4f} | Val R2: {r2_va:.4f}")

plt.tight_layout()
plt.show()


## 4. Multivariate Polynomial Regression

When our dataset features multiple input variables (e.g., two inputs $x_1$ and $x_2$), `PolynomialFeatures` generates terms corresponding to the degree of each variable, as well as **interaction terms** showing how the variables combine.

### 4.1 What are Interaction Terms?
Interaction terms (like $x_1 \cdot x_2$) allow the model to capture effects where the combined impact of two variables differs from their individual actions.

For example, if we have two inputs $[x_1, x_2]$ and set `degree=2`, the engineered outputs are:

$$\text{Transformed Features} = [1, x_1, x_2, x_1^2, x_2^2, x_1 x_2]$$

### 4.2 Combinatorial Expansion (Curse of Dimensionality)
As the number of features $m$ or the polynomial degree $d$ increases, the number of transformed features $N_f$ grows rapidly according to the combination formula:

$$N_f = \frac{(m + d)!}{m! \cdot d!}$$

| Features ($m$) | Degree ($d$) | Engineered Features ($N_f$) |
| :--- | :--- | :--- |
| 2 | 2 | 6 (including intercept) |
| 5 | 2 | 21 |
| 5 | 3 | 56 |
| 10 | 3 | 286 |

Because of this combinatorial growth, applying high-degree multivariate polynomial regression to datasets with many features will quickly cause models to overfit and slow down training significantly.

In [ ]:
# Generate synthetic 3D dataset: y = 0.5 * x1^2 - 1.2 * x2^2 + 0.8 * x1 * x2 + noise
np.random.seed(42)
x1 = np.random.uniform(-2, 2, 100)
x2 = np.random.uniform(-2, 2, 100)
y_3d = 0.5 * (x1**2) - 1.2 * (x2**2) + 0.8 * (x1 * x2) + np.random.normal(0, 0.2, 100)

df = pd.DataFrame({'x1': x1, 'x2': x2})

# 1. Apply PolynomialFeatures degree=2
poly_mult = PolynomialFeatures(degree=2, include_bias=False)
X_mult_poly = poly_mult.fit_transform(df)

print("Engineered Multivariate Features names:")
print(poly_mult.get_feature_names_out(['x1', 'x2']))

# 2. Train OLS Linear Regression on polynomial features
model_3d = LinearRegression()
model_3d.fit(X_mult_poly, y_3d)

# 3. Plotting the 3D regression surface
fig = plt.figure(figsize=(11, 8.5))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot of train data
ax.scatter(x1, x2, y_3d, color='#e74c3c', s=40, edgecolors='k', alpha=0.8, label='Data Points')

# Create grid for predictions
x1_grid = np.linspace(-2.2, 2.2, 30)
x2_grid = np.linspace(-2.2, 2.2, 30)
X1_mesh, X2_mesh = np.meshgrid(x1_grid, x2_grid)

# Flatten mesh to predict
grid_df = pd.DataFrame({'x1': X1_mesh.ravel(), 'x2': X2_mesh.ravel()})
grid_poly = poly_mult.transform(grid_df)
Y_pred_mesh = model_3d.predict(grid_poly).reshape(X1_mesh.shape)

# Plot fitted curved surface
surf = ax.plot_surface(
    X1_mesh, X2_mesh, Y_pred_mesh, 
    cmap='viridis', alpha=0.45, edgecolor='none', antialiased=True
)

ax.set_title("3D Multivariate Polynomial Regression Fitted Surface (Degree 2)", fontsize=12, fontweight='bold')
ax.set_xlabel('Feature x1')
ax.set_ylabel('Feature x2')
ax.set_zlabel('Target y')
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, label='Predicted Target Value')
ax.legend()

# Adjust viewing angle for better surface rendering
ax.view_init(elev=20, azim=45)
plt.show()


## Summary Checklist for Polynomial Regression

1. **Check for Non-Linearity:** Visualize features vs. target beforehand. If scatter plots display curves, simple linear regression will underfit.
2. **Begin with Degree 2:** Use degree 2 as a baseline. Only increase the degree if validation metrics ($R^2$, MSE) show significant improvements without overfitting.
3. **Use Cross-Validation:** To select the optimal degree, split data into training and test/validation sets (or use K-Fold cross-validation) to evaluate performance on unseen data.
4. **Apply Regularization for High Degrees:** If a higher degree is necessary to fit complex trends, consider utilizing **Ridge** or **Lasso** regression instead of standard OLS. They penalize extreme coefficients, preventing overfitting.
5. **Beware of Extrapolation:** High-degree polynomials behave extremely unpredictably outside the boundaries of the training dataset. Avoid using polynomial regression to make predictions outside the range of your training values.